# 08 — Extracción de frecuencias POS (Stanza)

Equivalente al notebook `10_extraer_frecuencias_POS.ipynb` de Karen.

Extrae lemas de verbos, adjetivos y sustantivos del subcorpus de salud usando Stanza.

**Entrada:** `salud_tweets_final.parquet`, `corpus_cleaned.parquet`  
**Salida:** `verbos_salud_stanza.parquet`, `adjetivos_salud_stanza.parquet`, `sustantivos_salud_stanza.parquet`

In [1]:
# ============================================================
# CELL 0 — CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosPropios')

print('[CONFIG] OK')
print(f'  DATA_PROCESSED : {DATA_PROCESSED.resolve()}')


[CONFIG] OK
  DATA_PROCESSED : C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosPropios


In [2]:
# ============================================================
# CELL 1 — IMPORTS Y CARGA
# Equivalente a celdas 0-2 del paper
# ============================================================
import pandas as pd
import stanza
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Subcorpus de salud (salida de NB04)
# Tiene columnas: chunk_id, id_doc, texto_chunk, Salud_*, score_max, subcat_max,
#                 etiqueta_salud, categoria_detectada
tweets_salud = pd.read_parquet(DATA_PROCESSED / 'salud_tweets_final.parquet')

# Columnas de interes (equiv. al df_filtrado de Karen)
cols_base = ['chunk_id', 'id_doc', 'texto_chunk',
             'etiqueta_salud', 'categoria_detectada', 'subcat_max']
cols_base = [c for c in cols_base if c in tweets_salud.columns]

df_salud = tweets_salud[cols_base].copy()

print(f'Tweets de salud: {len(df_salud):,}')
print(f'Columnas       : {list(df_salud.columns)}')
df_salud.head(2)


Tweets de salud: 29,230
Columnas       : ['chunk_id', 'id_doc', 'texto_chunk', 'etiqueta_salud', 'categoria_detectada', 'subcat_max']


,chunk_id,id_doc,texto_chunk,etiqueta_salud,categoria_detectada,subcat_max
0,4_1,4,"Normas que deberán cumplirse en las empresas, ...",1,Salud_Epidemiologia,Salud_Epidemiologia
1,6_1,6,La otra semana ya saldrá el aumento de contagi...,1,Salud_Lucha_enfermedades,Salud_Lucha_enfermedades


In [3]:
# ============================================================
# CELL 2 — DESCARGAR MODELO STANZA (solo primera vez)
# ============================================================
# stanza.download('es')   # descomentar si es la primera ejecución
print('Stanza listo. Si es la primera vez, descomenta stanza.download("es") arriba.')


Stanza listo. Si es la primera vez, descomenta stanza.download("es") arriba.


In [4]:
# ============================================================
# CELL 3 — INICIALIZAR PIPELINE
# ============================================================
nlp_stanza = stanza.Pipeline(
    lang="es",
    processors="tokenize,pos,lemma",
    use_gpu=False   # cambiar a True si hay GPU disponible
)
print('Pipeline Stanza inicializado.')


2026-06-02 20:15:59 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-06-02 20:16:00 INFO: Downloaded file to C:\Users\afpue\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-06-02 20:16:00 WARNING: Language es package default expects mwt, which has been added
2026-06-02 20:16:01 INFO: Loading these models for language: es (Spanish):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-06-02 20:16:01 INFO: Using device: cpu
2026-06-02 20:16:01 INFO: Loading: tokenize
2026-06-02 20:16:03 INFO: Loading: mwt
2026-06-02 20:16:03 INFO: Loading: pos
2026-06-02 20:16:05 INFO: Loading: lemma
2026-06-02 20:16:06 INFO: Done loading processors!


Pipeline Stanza inicializado.


In [5]:
# ============================================================
# CELL 4 — FUNCIÓN DE EXTRACCIÓN POS
# (Replica exacta de Karen)
# ============================================================
def extraer_pos_frecuencias_stanza(textos):
    """
    Procesa una lista de textos con Stanza.
    Devuelve tres listas de dicts {lema: frecuencia}:
      verbos_frec, adjetivos_frec, sustantivos_frec
    """
    verbos_frec, adjetivos_frec, sustantivos_frec = [], [], []

    for texto in tqdm(textos, desc="POS Stanza"):
        try:
            if not isinstance(texto, str) or not texto.strip():
                verbos_frec.append({})
                adjetivos_frec.append({})
                sustantivos_frec.append({})
                continue

            doc = nlp_stanza(texto)
            verbos, adjetivos, sustantivos = [], [], []

            for sent in doc.sentences:
                for w in sent.words:
                    if not w.lemma or not w.lemma.isalpha():
                        continue
                    upos = w.upos
                    lema = w.lemma.lower()
                    if upos == "VERB":
                        verbos.append(lema)
                    elif upos == "ADJ":
                        adjetivos.append(lema)
                    elif upos == "NOUN":
                        sustantivos.append(lema)

            verbos_frec.append(dict(Counter(verbos)))
            adjetivos_frec.append(dict(Counter(adjetivos)))
            sustantivos_frec.append(dict(Counter(sustantivos)))

        except Exception as e:
            print(f"Error: {e}")
            verbos_frec.append({})
            adjetivos_frec.append({})
            sustantivos_frec.append({})

    return verbos_frec, adjetivos_frec, sustantivos_frec


In [6]:
# ============================================================
# CELL 5 — EJECUTAR EXTRACCIÓN
# (Puede tardar 30-60 min dependiendo del hardware)
# ============================================================

# Karen usa 'texto_chunk' — aqui tambien
textos = df_salud['texto_chunk'].tolist()
verbos_frec, adjetivos_frec, sustantivos_frec = extraer_pos_frecuencias_stanza(textos)

print(f'Extraccion completada para {len(textos):,} tweets.')


POS Stanza: 100%|██████████| 29230/29230 [2:05:59<00:00,  3.87it/s]  

Extraccion completada para 29,230 tweets.


In [7]:
# ============================================================
# CELL 6 — CONSTRUIR DATAFRAMES Y GUARDAR
# Equivalente a celdas 7-9 del paper
# Karen guarda xlsx; aqui guardamos parquet (mejor para dicts)
# ============================================================

# Verbos
df_verbos = df_salud[cols_base].copy()
df_verbos['verbos_lemas_frecuencias'] = verbos_frec
df_verbos.to_parquet(DATA_PROCESSED / 'verbos_salud_stanza.parquet', index=False)
print('[GUARDADO] verbos_salud_stanza.parquet')

# Adjetivos
df_adjetivos = df_salud[cols_base].copy()
df_adjetivos['adjetivos_lemas_frecuencias'] = adjetivos_frec
df_adjetivos.to_parquet(DATA_PROCESSED / 'adjetivos_salud_stanza.parquet', index=False)
print('[GUARDADO] adjetivos_salud_stanza.parquet')

# Sustantivos
df_sustantivos = df_salud[cols_base].copy()
df_sustantivos['sustantivos_lemas_frecuencias'] = sustantivos_frec
df_sustantivos.to_parquet(DATA_PROCESSED / 'sustantivos_salud_stanza.parquet', index=False)
print('[GUARDADO] sustantivos_salud_stanza.parquet')


[GUARDADO] verbos_salud_stanza.parquet
[GUARDADO] adjetivos_salud_stanza.parquet
[GUARDADO] sustantivos_salud_stanza.parquet


In [8]:
# ============================================================
# CELL 7 — VERIFICACIÓN RÁPIDA
# ============================================================
# Top 20 verbos más frecuentes en el subcorpus de salud
from collections import Counter

total_verbos = Counter()
for d in verbos_frec:
    total_verbos.update(d)

total_sust = Counter()
for d in sustantivos_frec:
    total_sust.update(d)

total_adj = Counter()
for d in adjetivos_frec:
    total_adj.update(d)

print('Top 20 VERBOS:')
print(total_verbos.most_common(20))
print()
print('Top 20 SUSTANTIVOS:')
print(total_sust.most_common(20))
print()
print('Top 20 ADJETIVOS:')
print(total_adj.most_common(20))

print()
print('Notebook 08 completado.')
print('Siguiente -> 09_descriptivos_subcategorias_salud.ipynb')


Top 20 VERBOS:
[('tener', 3563), ('haber', 2653), ('hacer', 2249), ('dar', 1470), ('ir', 1374), ('seguir', 1342), ('decir', 1170), ('evitar', 1105), ('realizar', 1054), ('registrar', 998), ('llegar', 931), ('confirmar', 859), ('reportar', 827), ('saber', 752), ('ver', 749), ('informar', 726), ('morir', 680), ('prevenir', 614), ('pasar', 614), ('salir', 597)]

Top 20 SUSTANTIVOS:
[('caso', 7259), ('contagio', 4454), ('coronavirus', 4176), ('pandemia', 3425), ('virus', 2682), ('vacuna', 2473), ('persona', 2456), ('muerte', 2417), ('día', 1895), ('salud', 1848), ('paciente', 1795), ('país', 1778), ('medida', 1722), ('prueba', 1378), ('enfermedad', 1120), ('cifra', 1028), ('año', 953), ('brote', 951), ('hospital', 890), ('número', 883)]

Top 20 ADJETIVOS:
[('nuevo', 4930), ('total', 1611), ('positivo', 1558), ('recuperado', 1493), ('fallecido', 1203), ('primero', 1117), ('contagiado', 1074), ('sanitario', 1040), ('confirmado', 913), ('mayor', 862), ('último', 825), ('chino', 764), ('públic